In [0]:
df = spark.read.format("excel") \
    .option("header", "true") \
    .load("/Volumes/workspace/default/vol_clientes/Base trabajo final.xlsx")

display(df)


In [0]:
# borrar tablas si existen (para poder ejecutar desde cero)
spark.sql("DROP TABLE IF EXISTS bronze_clientes")
spark.sql("DROP TABLE IF EXISTS silver_clientes")
spark.sql("DROP TABLE IF EXISTS gold_clientes")
spark.sql("DROP TABLE IF EXISTS clientes_scoring")

In [0]:
df.write.mode("overwrite") \
  .option("overwriteSchema","true") \
  .saveAsTable("bronze_clientes")

In [0]:
df2 = spark.sql("SELECT * FROM bronze_clientes")
display(df2)

In [0]:
from pyspark.sql.functions import col

df = spark.read.table("bronze_clientes")

# eliminar duplicados
df_clean = df.dropDuplicates()

# reemplazar nulos por 0
df_clean = df_clean.fillna(0)

display(df_clean)

In [0]:
df = spark.read.format("excel") \
    .option("inferSchema","true") \
    .load("/Volumes/workspace/default/vol_clientes/Base trabajo final.xlsx")

display(df)


In [0]:
pdf = df.toPandas()
pdf.head()

In [0]:
pdf.columns = pdf.iloc[0]
pdf = pdf[1:]
pdf.head()

In [0]:
df = spark.createDataFrame(pdf)
display(df)

In [0]:
# convertir nombres a formato limpio (snake_case) hay espacios
import re

def limpiar_nombre(col):
    col = col.lower()
    col = col.strip()
    col = col.replace(" ", "_")
    col = re.sub(r'[^a-zA-Z0-9_]', '', col)
    return col

df = df.toDF(*[limpiar_nombre(c) for c in df.columns])

df.columns

In [0]:
df.write.mode("overwrite")\
.option("overwriteSchema","true")\
.saveAsTable("bronze_clientes")

In [0]:
df_clean = df.dropDuplicates().fillna(0)

df_clean.write.mode("overwrite") \
  .option("overwriteSchema","true") \
  .saveAsTable("silver_clientes")

In [0]:
spark.read.table("silver_clientes").columns

In [0]:
pdf = spark.read.table("silver_clientes").toPandas()
pdf.head()

In [0]:
import pandas as pd

In [0]:
cols_numericas = [
    "edad",
    "ingresos",
    "monto_credito",
    "cuota_mensual",
    "dias_mora",
    "cuotas_en_mora",
    "uso_credito",
    "cupo",
    "incumplimiento"
]

for col in cols_numericas:
    pdf[col] = pd.to_numeric(pdf[col], errors="coerce")

pdf = pdf.fillna(0)

pdf.dtypes

In [0]:
X = pdf[[
    "edad",
    "ingresos",
    "monto_credito",
    "cuota_mensual",
    "dias_mora",
    "cuotas_en_mora",
    "uso_credito",
    "cupo"
]]

y = pdf["incumplimiento"]

In [0]:
from sklearn.model_selection import train_test_split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [0]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression()

In [0]:
modelo.fit(X_train, y_train)

In [0]:
y_pred = modelo.predict(X_test)

In [0]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [0]:
pdf["score_riesgo"] = modelo.predict_proba(X)[:,1]
pdf.head()

In [0]:
gold_df = spark.createDataFrame(pdf)

In [0]:
gold_df.write.mode("overwrite").saveAsTable("gold_scoring_clientes")

In [0]:
pdf = spark.read.table("gold_scoring_clientes").toPandas()

In [0]:
pdf.to_csv("/Volumes/workspace/default/vol_clientes/scoring.csv", index=False)